In [1]:
print("hello")

hello


In [2]:
import pandas as pd 
import numpy as np 
from pprint import pprint


In [3]:
# importing .txt 
# from gtfs_data 
# sth to do wuth memory error 

df_agency = pd.read_csv("../../gtfs_data/agency.txt")
df_calendar_dates = pd.read_csv("../../gtfs_data/calendar_dates.txt")
df_calendar = pd.read_csv("../../gtfs_data/calendar.txt")
df_levels = pd.read_csv("../../gtfs_data/levels.txt")
df_notes = pd.read_csv("../../gtfs_data/notes.txt")
df_pathways = pd.read_csv("../../gtfs_data/pathways.txt")
df_routes = pd.read_csv("../../gtfs_data/routes.txt")
df_shapes = pd.read_csv("../../gtfs_data/shapes.txt")
df_stop_times = pd.read_csv("../../gtfs_data/stop_times.txt")
df_stops = pd.read_csv("../../gtfs_data/stops.txt")
df_trips = pd.read_csv("../../gtfs_data/trips.txt")


C:\Users\axbou\AppData\Local\Temp\ipykernel_8664\762058471.py:13: DtypeWarning: Columns (0: trip_id, 1: stop_id, 2: stop_headsign, 3: stop_note) have mixed types. Specify dtype option on import or set low_memory=False.
  df_stop_times = pd.read_csv("../../gtfs_data/stop_times.txt")
C:\Users\axbou\AppData\Local\Temp\ipykernel_8664\762058471.py:14: DtypeWarning: Columns (0: platform_code) have mixed types. Specify dtype option on import or set low_memory=False.
  df_stops = pd.read_csv("../../gtfs_data/stops.txt")
C:\Users\axbou\AppData\Local\Temp\ipykernel_8664\762058471.py:15: DtypeWarning: Columns (0: trip_note) have mixed types. Specify dtype option on import or set low_memory=False.
  df_trips = pd.read_csv("../../gtfs_data/trips.txt")


In [4]:
# retrive a list of corresponding df_* with there columns in a list agency_columns = df_agency.columns.tolist()
table_columns = {
"agency": [],
"calendar": [],
"calendar_dates": [],
"levels": [],
"notes": [],
"pathways": [],
"routes": [],
"shapes": [],
"stop_times": [],
"stops": [],
"trips": []
}
dataframes = {"df_agency": df_agency, "df_calendar_dates": df_calendar_dates, "df_calendar": df_calendar, "df_levels": df_levels, "df_notes": df_notes, "df_pathways": df_pathways, "df_routes": df_routes, "df_shapes": df_shapes, "df_stop_times": df_stop_times, "df_stops": df_stops, "df_trips": df_trips}
for name, df in dataframes.items():
    table_columns[name.split("df_")[1]] = df.columns.tolist()


In [5]:
# dumping the table_columns dict to a json file
import json
with open("table_columns.json", "w") as f:
    json.dump(table_columns, f, indent=4)

In [6]:
import json

schema = table_columns # your dictionary

relationships = []

for table, columns in schema.items():
    for col in columns:
        if col.endswith("_id"):
            target_table = col.replace("_id", "") + "s"

            if target_table in schema:
                relationships.append((table, col, target_table, col))

print(relationships)

[('levels', 'level_id', 'levels', 'level_id'), ('notes', 'note_id', 'notes', 'note_id'), ('pathways', 'pathway_id', 'pathways', 'pathway_id'), ('routes', 'route_id', 'routes', 'route_id'), ('shapes', 'shape_id', 'shapes', 'shape_id'), ('stop_times', 'trip_id', 'trips', 'trip_id'), ('stop_times', 'stop_id', 'stops', 'stop_id'), ('stops', 'stop_id', 'stops', 'stop_id'), ('stops', 'level_id', 'levels', 'level_id'), ('trips', 'route_id', 'routes', 'route_id'), ('trips', 'trip_id', 'trips', 'trip_id'), ('trips', 'shape_id', 'shapes', 'shape_id')]


In [7]:
import sys
print(sys.executable)

c:\Users\Public\transport\.venv\Scripts\python.exe


In [11]:
def to_dbml(schema):
    lines = []

    for table, columns in schema.items():
        lines.append(f"Table {table} {{")
        for col in columns:
            if col.endswith("_id") and col != f"{table[:-1]}_id":
                lines.append(f"  {col} varchar [ref: > {col[:-3]}s.{col}]")
            else:
                lines.append(f"  {col} varchar")
        lines.append("}\n")

    return "\n".join(lines)

print(to_dbml(table_columns))
with open("schema.dbml", "w") as f:
    f.write(to_dbml(table_columns))

Table agency {
  agency_id varchar [ref: > agencys.agency_id]
  agency_name varchar
  agency_url varchar
  agency_timezone varchar
  agency_lang varchar
  agency_phone varchar
}

Table calendar {
  service_id varchar [ref: > services.service_id]
  monday varchar
  tuesday varchar
  wednesday varchar
  thursday varchar
  friday varchar
  saturday varchar
  sunday varchar
  start_date varchar
  end_date varchar
}

Table calendar_dates {
  service_id varchar [ref: > services.service_id]
  date varchar
  exception_type varchar
}

Table levels {
  level_id varchar
  level_index varchar
  level_name varchar
}

Table notes {
  note_id varchar
  note_text varchar
}

Table pathways {
  pathway_id varchar
  from_stop_id varchar [ref: > from_stops.from_stop_id]
  to_stop_id varchar [ref: > to_stops.to_stop_id]
  pathway_mode varchar
  is_bidirectional varchar
  traversal_time varchar
}

Table routes {
  route_id varchar
  agency_id varchar [ref: > agencys.agency_id]
  route_short_name varchar
  r